# D1-04 First LCA from demand to score in Brightway

This notebook is the main Day 1 milestone: we bootstrap the course project from a prepared Brightway archive, import the bundled BAFU workbook, inspect the resulting database, and run a first LCA.

The hands-on exercises use BAFU throughout the course. We use `ecoinvent-3.10-biosphere` only as a convenient starting project because it already contains `biosphere3` and a populated set of LCIA methods.


## Learning goals

- Install or switch to the course `brightway` project from a prepared archive.
- Import the bundled BAFU workbook with `bw2io.ExcelImporter`.
- Understand the role of strategies, biosphere matching, statistics, and database writing.
- Inspect the LCIA methods that are already available in the project.
- Inspect the imported database by searching for activities and exchanges.
- Run a first LCA score with an explicitly chosen activity and method.
- Recognize how an `ecoinvent` import would fit into the same overall workflow.


## Background references

- Mutel, C. (2017). *Brightway: An open source framework for life cycle assessment*. Journal of Open Source Software, 2(12), 236. https://doi.org/10.21105/joss.00236
- Database of the Swiss Federal Administration, FOEN:20XY, Federal Office for the Environment, 2025.


## 1) Install or switch to the course project

We start from a prepared Brightway project archive instead of building `biosphere3` and the LCIA methods from scratch.

The call `bi.remote.install_project('ecoinvent-3.10-biosphere', project_name)` downloads the archive on first use, restores it as a new project, and gives us a project that already contains `biosphere3` and many LCIA methods.
If the project already exists locally, we simply switch to it.


In [17]:
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
import bw2calc as bc
import bw2data as bd
import bw2io as bi

# this is just to let dataframe print fully
pd.set_option('display.max_colwidth', None)

In [18]:
bd.projects

Brightway2 projects manager with 17 objects:
	aalborg-architecture-demo
	aalborg-check
	aalborg-rlcia-2026
	aalborg-rlcia-sandbox
	act_templates_v311
	default
	demo
	metafuels_lca_v391
	metafuels_project
	saf_lca_v312
	space_ei311_ESA130b_remind
	space_lca_v391
	space_v311
	sssd_v103
	sssd_v311
	sssdv103_ev311
	tprise
Use `projects.report()` to get a report on all projects.

In [19]:
# this is how you can delete a project, if need be
# bd.projects.delete_project("aalborg-rlcia-2026", delete_dir=True)

In [20]:
# let's choose a project name
project_name = "aalborg-rlcia-2026"
project_key = "ecoinvent-3.10-biosphere"
# and make a list of existing projects
existing_projects = [project.name for project in bd.projects]

In [21]:
# if the project is already part of the projects' list, then our project already exists
# in this case, we only need to "switch" to it
if project_name in existing_projects:
    bd.projects.set_current(project_name)
    print('Switched to existing project:', bd.projects.current)

# if not, then we create it
else:
    bi.remote.install_project("ecoinvent-3.10-biosphere", project_name)
    # and we switch to it
    bd.projects.set_current(project_name)
    print('Installed and switched to project:', bd.projects.current)

Switched to existing project: aalborg-rlcia-2026


In [22]:
# this is needed later when importing LCI databases
bi.create_core_migrations()

print('Databases available now:', list(bd.databases))
print('LCIA methods available now:', len(bd.methods))

Databases available now: ['ecoinvent-3.10-biosphere', 'bafu']
LCIA methods available now: 668


Let's check the size of `ecoinvent-3.10-biosphere`

In [23]:
len(bd.Database("ecoinvent-3.10-biosphere"))

4362

The prepared project already contains the core biosphere database and a large set of LCIA methods.
So the hands-on import work in this notebook is focused on the BAFU technosphere database.


## 2) Confirm the bundled BAFU workbook

The BAFU database we will be using is available as a workbook, contained in this repository:

- `data/lci-bafu.xlsx`


In [24]:
bafu_file = Path('../../data/lci-bafu.xlsx')
print('Workbook path:', bafu_file.resolve())
print('Found:', bafu_file.exists())
if bafu_file.exists():
    print('Workbook size [MB]:', round(bafu_file.stat().st_size / 1024 / 1024, 2))


Workbook path: C:\Users\treyer_k\Documents\GitHub\aalborg-regionalized-lcia\data\lci-bafu.xlsx
Found: True
Workbook size [MB]: 29.47


## 3) Import pipeline: `ExcelImporter` -> strategies -> biosphere matching -> statistics -> database writing

This is the import sequence to understand in this notebook:

1. Create an `ExcelImporter` object from the workbook path.
2. Apply the standard import strategies (data cleanup, mostly).
3. Match biosphere flows against `biosphere3`.
4. Check import statistics and unlinked exchanges.
5. Write the database to the current project.


In [25]:
database_name = 'bafu'

# create the ExcelImporter object
importer = bi.ExcelImporter(str(bafu_file))
print('Importer type:', type(importer).__name__)
print('Raw datasets loaded:', len(importer.data))


Extracted 1 worksheets in 104.14 seconds
Importer type: ExcelImporter
Raw datasets loaded: 11747


In [26]:
# data cleanup
importer.apply_strategies()
print('Applied import strategies.')

Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_restore_temporal_distributions
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 15 strategies in 10.83 seconds
Applied import strategies.


In [27]:
importer.match_database(fields=('name', 'reference product', 'location'))

Applying strategy: link_iterable_by_fields


In [28]:
# now, we try to match the biosphere exchanges with the flows of the `biosphere` database
# on the basis of 'name', 'unit', 'categories'
importer.match_database('ecoinvent-3.10-biosphere', fields=('name', 'unit', 'categories'))
print('Matched biosphere exchanges against biosphere3.')

Applying strategy: link_iterable_by_fields
Matched biosphere exchanges against biosphere3.


In [29]:
# we check the status of exchanges linking
importer.statistics()

Graph statistics for `bafu` importer:
11747 graph nodes:
	processwithreferenceproduct: 11747
408975 graph edges:
	biosphere: 284445
	technosphere: 112783
	production: 11747
408975 edges to the following databases:
	ecoinvent-3.10-biosphere: 284445
	bafu: 124530
0 unique unlinked edges (0 total):




(11747, 408975, 0, 0)

`0 unique unlinked edges (0 total)` is usually the sweet message you want to see.  
This means we can go ahead and write the database to the project.

In [30]:
importer.write_database()

15:10:47+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 11747/11747 [00:19<00:00, 597.25it/s] 


15:11:09+0200 [info     ] Vacuuming database            
Created database: bafu


*Remark*:
This is the general way how to import inventories from excel files, as long as the latter come with the correct format bw can read.
In case the importer.statistics() show unlinked exchanges, you can make them visible with the following code. 

In [31]:
list(importer.unlinked)
#or
importer.write_excel() #this gives you a new Excel file with your complete inventory, where the unlinked exchanges are highlighted in red, so you can check them and fix them if needed
#or
importer.write_excel(only_unlinked=True) #this gives you a new Excel file with only the unlinked exchanges, so you can check them and fix them if needed

Wrote matching file to:
C:\Users\treyer_k\AppData\Local\pylca\Brightway3\aalborg-rlcia-2026.28f37817\output\db-matching-bafu.xlsx
Wrote matching file to:
C:\Users\treyer_k\AppData\Local\pylca\Brightway3\aalborg-rlcia-2026.28f37817\output\db-matching-bafu-unlinked.xlsx


WindowsPath('C:/Users/treyer_k/AppData/Local/pylca/Brightway3/aalborg-rlcia-2026.28f37817/output/db-matching-bafu-unlinked.xlsx')

## 4) Inspect the LCIA methods already available in the project

Because the project was installed from the `ecoinvent-3.10-biosphere` archive, the LCIA methods are already in place. This is not just a convenience: an LCIA method in `brightway` is a list of `(biosphere flow, characterization factor)` pairs that points directly at the flows of one specific biosphere database, so methods and biosphere flows only make sense together and are therefore distributed together.

The factors themselves come from `ecoinvent`, which publishes an *LCIA Implementation* workbook with every release, listing the characterization factors of all supported methods (EF, ReCiPe, IPCC, ...) already mapped onto ecoinvent's own elementary flows. `bw2io` reads that workbook, matches each row to a flow in the biosphere database by name and compartment, and registers the result as a `bd.Method` — and it refuses to do so if no populated biosphere database is present, which is the dependency above enforced in code.


In [32]:
preferred_method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

print('Databases available:', list(bd.databases))
print('Number of LCIA methods available:', len(bd.methods))
print('Preferred climate-change method available:', preferred_method in bd.methods)


Databases available: ['ecoinvent-3.10-biosphere', 'bafu']
Number of LCIA methods available: 668
Preferred climate-change method available: True


Let's quickly inspect a few available methods before choosing one for the first LCA.


In [39]:
sample_methods = sorted(bd.methods)[:10]
climate_methods = [m for m in sorted(bd.methods) if 'climate change' in ' | '.join(m).lower()]  #This is a list comprehension search. It is often applied to search through biosphere, databases, or methods. 

print('First 10 LCIA methods in the project:')
for method in sample_methods:
    print('-', ' | '.join(method))

print('\nFirst climate-related methods:')
for method in climate_methods[:10]:
    print('-', ' | '.join(method))


First 10 LCIA methods in the project:
- CML v4.8 2016 | acidification | acidification (incl. fate, average Europe total, A&B)
- CML v4.8 2016 | climate change | global warming potential (GWP100)
- CML v4.8 2016 | ecotoxicity: freshwater | freshwater aquatic ecotoxicity (FAETP inf)
- CML v4.8 2016 | ecotoxicity: marine | marine aquatic ecotoxicity (MAETP inf)
- CML v4.8 2016 | ecotoxicity: terrestrial | terrestrial ecotoxicity (TETP inf)
- CML v4.8 2016 | energy resources: non-renewable | abiotic depletion potential (ADP): fossil fuels
- CML v4.8 2016 | eutrophication | eutrophication (fate not incl.)
- CML v4.8 2016 | human toxicity | human toxicity (HTP inf)
- CML v4.8 2016 | material resources: metals/minerals | abiotic depletion potential (ADP): elements (ultimate reserves)
- CML v4.8 2016 | ozone depletion | ozone layer depletion (ODP steady state)

First climate-related methods:
- CML v4.8 2016 | climate change | global warming potential (GWP100)
- CML v4.8 2016 no LT | climate ch

### REMINDER: how to inspect an LCIA method?

In [40]:
my_method = bd.methods.random()
my_method

('EF v3.0',
 'human toxicity: non-carcinogenic, inorganics',
 'comparative toxic unit for human (CTUh)')

In [41]:
method_obj = bd.Method(my_method)
method_data = method_obj.load()

In [42]:
print('Method metadata:')
pprint(method_obj.metadata)

Method metadata:
{'abbreviation': 'ef-v30hc.a17bdbd8893590cd7965105a2fb0bacc',
 'database': 'ecoinvent-3.10-biosphere',
 'ecoinvent_version': '3.10',
 'filepath': '/Users/cmutel/Library/Application '
             'Support/EcoinventInterface/cache/ecoinvent '
             '3.10_LCIA_implementation/LCIA Implementation 3.10.xlsx',
 'geocollections': ['world'],
 'num_cfs': 170,
 'unit': 'CTUh'}


In [43]:
# an empty list, to store the results
lcia_data = []

for flow_id, cf in method_data:
    flow = bd.get_activity(flow_id)
    lcia_data.append(
        [
            flow["name"],
            cf
        ]
    )

pd.DataFrame(lcia_data, columns=["flow name", "cf"])

,flow name,cf
0,Allyl chloride,3.897700e-08
1,Aluminium hydroxide,1.765100e-07
2,Ammonia,1.526900e-09
3,Ammonia,1.326200e-08
4,Ammonia,2.499700e-08
...,...,...
165,Trichlorosilane,3.938500e-08
166,Trichlorosilane,2.672400e-10
167,Trichlorosilane,3.938500e-08
168,Trichlorosilane,1.982600e-08


## 5) Import your own LCIA method

We will use bw2io.LCIAExcelImporter to import a user-defined LCIA method file.  
You can find those files under `data/LCIA methods`. We will import `01__acidification__accumulated_exceedance_ae.xlsx`.

In [44]:
importer = bi.ExcelLCIAImporter(
    "../../data/LCIA methods/01__acidification__accumulated_exceedance_AE.xlsx",
    name=("EF 3.0", "Acidification"),
    description="Some method about ocean acidification",
    unit="kg SO2-eq."
)

In [45]:
# data cleanup
importer.apply_strategies()

Applying strategy: csv_restore_tuples
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: set_biosphere_type
Applying strategy: drop_unspecified_subcategories
Applying strategy: link_iterable_by_fields
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applied 8 strategies in 0.12 seconds


In [46]:
# let's check if we have any unlinked CFs
importer.statistics()

1 methods
19 cfs
0 unlinked cfs


(1, 19, 0)

In [47]:
# we're good to go!
importer.write_methods()

Wrote 1 LCIA methods with 19 characterization factors


In [48]:
# let's confirm that our method is available
("EF 3.0", "Acidification") in bd.methods

True

In [49]:
# let's inspect it
acid_method = bd.Method(("EF 3.0", "Acidification"))
pprint(acid_method.metadata)

{'abbreviation': 'ef-30a.33e899d02d0b2714248f9d3f8c74391d',
 'description': 'Some method about ocean acidification',
 'filename': '01__acidification__accumulated_exceedance_AE.xlsx',
 'geocollections': ['world'],
 'num_cfs': 19,
 'unit': 'kg SO2-eq.'}


In [50]:
for flow_id, cf in acid_method.load():
    flow = bd.get_activity(flow_id)
    print(flow["name"], cf)

Ammonia 3.02
Nitrogen oxides 0.74
Nitrogen oxides 0.74
Ammonia 3.02
Sulfur trioxide 1.04821
Sulfur trioxide 1.04821
Sulfur trioxide 1.04821
Sulfur dioxide 1.31
Sulfur dioxide 1.31
Ammonia 3.02
Nitrogen oxides 0.74
Sulfur dioxide 1.31
Sulfur dioxide 1.31
Nitrogen oxides 0.74
Nitrogen oxides 0.74
Sulfur trioxide 1.04821
Sulfur oxides 1.31
Nitric oxide 1.13467
Ammonia 3.02


## 6) Inspect the imported database

Now we reconnect the import workflow from the previous notebooks: project -> database -> activity -> exchange.

A good way to start is to search the database by name, then refine with additional filters such as `location`.

In [51]:
db.search?

Object `db.search` not found.


In [52]:
db = bd.Database("bafu")

hits = db.search(
    "operation, passenger car",
    limit=5,
)
pd.DataFrame(
    [
        {
            'name': act['name'],
            'reference product': act['reference product'],
            'location': act.get('location'),
            'unit': act.get('unit'),
        }
        for act in hits
    ]
)

,name,reference product,location,unit
0,"Operation, passenger car, ethanol 5%","Operation, passenger car, ethanol 5%",CH,kilometer
1,"Operation, passenger car, natural gas","Operation, passenger car, natural gas",CH,kilometer
2,"Operation, passenger car, methanol","Operation, passenger car, methanol",CH,kilometer
3,"Operation, passenger car, rape seed methyl ester 5%","Operation, passenger car, rape seed methyl ester 5%",CH,kilometer
4,"Operation, passenger car, methane, 96 vol-%, from biogas","Operation, passenger car, methane, 96 vol-%, from biogas",CH,kilometer


In [53]:
db = bd.Database("bafu")
print('Number of activities in bafu:', len(db))

Number of activities in bafu: 11747


We can use `bw2data.search()` to look for activities (this, naturally, works also to search the `biosphere` database). A `search` gets you close quickly. But to be more explicit, you can iterate through the database yourself and combine several criteria in a list comprehension.

In [54]:
gasoline_car_candidates = [
    act
    for act in db
    if 'gasoline' in act['name'].lower()
    and any(term in act['reference product'].lower() for term in ['car', 'gasoline'])
    and act["location"] == "RER"
    and act["unit"] == "kilometer"
]

pd.DataFrame(
    [
        {
            'name': act['name'],
            'location': act.get('location'),
            'unit': act.get('unit'),
        }
        for act in gasoline_car_candidates[:10]
    ]
)

,name,location,unit
0,"Transport, passenger car, gasoline hybrid, Large, 2013, EURO-5",RER,kilometer
1,"Transport, passenger car, plugin gasoline hybrid, Medium, 2013, EURO-5",RER,kilometer
2,"Transport, passenger car, gasoline, Compact, 2021, EURO-6d",RER,kilometer
3,"Transport, passenger car, gasoline, Compact, 2019, EURO-6d-TEMP",RER,kilometer
4,"Transport, passenger car, gasoline, Large, 2018, EURO-6c",RER,kilometer
5,"Transport, passenger car, gasoline hybrid, Medium, 2018, EURO-6c",RER,kilometer
6,"Transport, passenger car, plugin gasoline hybrid, Large, 2013, EURO-5",RER,kilometer
7,"Transport, passenger car, plugin gasoline hybrid, Compact, 2016, EURO-6ab",RER,kilometer
8,"Transport, passenger car, gasoline, Compact, 2013, EURO-5",RER,kilometer
9,"Transport, passenger car, gasoline hybrid, Compact, 2021, EURO-6d",RER,kilometer


Geography filters are often more reliable than free-text search alone. For example, you can ask for passenger-car activities in Switzerland (`CH`), Europe (`RER`), or global datasets (`GLO`).

In [56]:
car_by_location = [
    act
    for act in db
    if 'passenger car' in act['name'].lower()
    and act.get('location') in {'CH', 'RER'}
    and act["unit"] == "kilometer"
]

pd.DataFrame(
    [
        {
            'name': act['name'],
            'location': act.get('location'),
            'unit': act.get('unit'),
        }
        for act in car_by_location[:15]
    ]
)

,name,location,unit
0,"Operation, passenger car, ethanol 5%",CH,kilometer
1,"Transport, passenger car, plugin gasoline hybrid, Compact, 2013, EURO-5",CH,kilometer
2,"Transport, passenger car, plugin diesel hybrid, Compact, 2016, EURO-6ab",CH,kilometer
3,"Transport, passenger car, battery electric, NMC-622 battery, label-certified electricity, Medium, 2021",CH,kilometer
4,"Transport, passenger car, plugin gasoline hybrid, label-certified electricity, Compact, 2019, EURO-6d-TEMP",CH,kilometer
5,"Transport, passenger car, diesel hybrid, Large SUV, 2018, EURO-6c",CH,kilometer
6,"Transport, passenger car, diesel, Large SUV, 2019, EURO-6d-TEMP",RER,kilometer
7,"Transport, passenger car, compressed gas, Large, 2016, EURO-6ab",RER,kilometer
8,"Transport, passenger car, plugin gasoline hybrid, UF=0%, Large SUV, 2019, EURO-6d-TEMP",CH,kilometer
9,"Transport, passenger car, plugin gasoline hybrid, Compact, 2013, EURO-5",RER,kilometer


Let's choose one activity relating to driving a gasoline car and inspect its metadata and exchanges. Here we look at an `operation` activity, because it gives a nice mix of technosphere input(s) and biosphere emissions.

In [59]:
selected_activity = gasoline_car_candidates[0]

pprint(selected_activity.as_dict())
#checking an activity .as_dict() can also help you to know which fields you can include in the list comprehension search.


{'code': 'bafu-514046',
 'comment': 'Driving cycle: WLTC. Combustion engine power: 110 [kW]. Electric '
            'motor power: 36 [kW]. Power share from combustion engine: 75 [%]. '
            'Km over lifetime: 200000 [km]. Yearly mileage: 12000 [km/year]. '
            'Autonomy on a full tank/battery: 747 [km]. Tank-to-wheel '
            'efficiency: 20 [%]. Tank-to-wheel energy consumption: 2751 '
            '[kj/km]. Fuel tank capacity: 571 [kWh]. Curb mass (excl. driver '
            'and cargo): 1843 [kg]. Driving mass (incl. driver and cargo): '
            '1983 [kg]..',
 'database': 'bafu',
 'filename': 'process_df8259f4-d27d-3322-87bc-86cdd3ae67f9.xml',
 'id': 343018000870277120,
 'location': 'RER',
 'name': 'Transport, passenger car, gasoline hybrid, Large, 2013, EURO-5',
 'reference product': 'Transport, passenger car, gasoline hybrid, Large, 2013, '
                      'EURO-5',
 'type': 'processwithreferenceproduct',
 'unit': 'kilometer',
 'worksheet name': 'bafu

Once you have one activity, you can iterate through its exchanges. Brightway lets you inspect `production()`, `technosphere()`, and `biosphere()` separately.

In [60]:
def preview_exchanges(exchanges, n=5):
    rows = []
    for exc in list(exchanges)[:n]:
        rows.append(
            {
                'input': exc.input['name'],
                'amount': exc['amount'],
                'unit': exc.input.get('unit'),
                'location': exc.input.get('location'),
                'categories': '::'.join(exc.input.get('categories') or ()),
                'type': exc['type'],
            }
        )
    return pd.DataFrame(rows)

In [61]:
print('Production exchanges')
preview_exchanges(selected_activity.production(), n=3)

Production exchanges


,input,amount,unit,location,categories,type
0,"Transport, passenger car, gasoline hybrid, Large, 2013, EURO-5",1,kilometer,RER,,production


In [62]:
print('Technosphere exchanges')
preview_exchanges(selected_activity.technosphere(), n=5)

Technosphere exchanges


,input,amount,unit,location,categories,type
0,"Brake wear emissions, passenger car",0.00001,kilogram,RER,,technosphere
1,"Disposal, road",0.00111,meter-year,RER,,technosphere
2,Fuel supply for gasoline vehicles,0.06560,kilogram,RER,,technosphere
3,"Maintenance, passenger car",0.00001,unit,RER,,technosphere
4,"Operation, maintenance, road",0.00129,meter-year,CH,,technosphere


In [63]:
print('Biosphere exchanges')
preview_exchanges(selected_activity.biosphere(), n=8)

Biosphere exchanges


,input,amount,unit,location,categories,type
0,1-Pentene,4.120000e-09,kilogram,None,air::non-urban air or from high stacks,biosphere
1,1-Pentene,1.370000e-09,kilogram,None,air::non-urban air or from high stacks,biosphere
2,1-Pentene,5.360000e-09,kilogram,None,air::urban air close to ground,biosphere
3,Acetaldehyde,3.290000e-08,kilogram,None,air::non-urban air or from high stacks,biosphere
4,Acetaldehyde,1.090000e-08,kilogram,None,air::non-urban air or from high stacks,biosphere
5,Acetaldehyde,4.290000e-08,kilogram,None,air::urban air close to ground,biosphere
6,Acetone,2.470000e-08,kilogram,None,air::non-urban air or from high stacks,biosphere
7,Acetone,8.210000e-09,kilogram,None,air::non-urban air or from high stacks,biosphere


You can use the same idea for any activity attributes (classifications, authors, year of publication), provided those are filled in. 

## Checkpoint: read an activity before calculating it

- **Predict:** Before selecting an activity, write down which exchange families you expect: one reference-product output, inputs from other activities, and direct environmental flows.
- **Run:** In the TODO cell, select one gasoline passenger-car operation activity, store it in `selected_activity`, and preview its production, technosphere, and biosphere exchanges. Refine by `location` if the search returns many candidates.
- **Interpret:** Name one exchange that defines the service, one that represents an upstream requirement, and one that represents a direct environmental intervention.
- **Check:** A normal operation activity should have a production exchange and non-empty technosphere and biosphere lists of exchanges. 

In [ ]:
# TODO
# Example pattern:
# hits = [
#     act for act in db
#     if 'passenger car' in act['name'].lower()
#     and any(term in act['name'].lower() for term in ['gasoline', 'petrol'])
# ]
# selected_activity = hits[0]
# preview_exchanges(selected_activity.technosphere(), n=3)
# preview_exchanges(selected_activity.biosphere(), n=3)

In [64]:
db = bd.Database("bafu")
hits = [
    act
    for act in db
    if 'passenger car' in act['name'].lower()
    and any(term in act['name'].lower() for term in ['gasoline', 'petrol'])
    and act['unit'] == "kilometer"
]

selected_activity = hits[0]

print('Selected activity:', selected_activity['name'])
print('Location:', selected_activity.get('location'))
print('Unit:', selected_activity.get('unit'))

Selected activity: Transport, passenger car, plugin gasoline hybrid, label-certified electricity, Large, 2018, EURO-6c
Location: CH
Unit: kilometer


In [65]:
print('First production exchanges:')
preview_exchanges(selected_activity.production(), n=3)

First production exchanges:


,input,amount,unit,location,categories,type
0,"Transport, passenger car, plugin gasoline hybrid, label-certified electricity, Large, 2018, EURO-6c",1,kilometer,CH,,production


In [66]:
print('First technosphere exchanges:')
preview_exchanges(selected_activity.technosphere(), n=3)

First technosphere exchanges:


,input,amount,unit,location,categories,type
0,"Brake wear emissions, passenger car",0.000010,kilogram,RER,,technosphere
1,"Carbon dioxide liquid, at plant",0.000008,kilogram,RER,,technosphere
2,"Disposal, road",0.001110,meter-year,RER,,technosphere


In [67]:
print('First biosphere exchanges:')
preview_exchanges(selected_activity.biosphere(), n=5)

First biosphere exchanges:


,input,amount,unit,location,categories,type
0,1-Pentene,5.380000e-10,kilogram,None,air::non-urban air or from high stacks,biosphere
1,1-Pentene,2.000000e-10,kilogram,None,air::non-urban air or from high stacks,biosphere
2,1-Pentene,1.270000e-09,kilogram,None,air::urban air close to ground,biosphere
3,Acetaldehyde,4.300000e-09,kilogram,None,air::non-urban air or from high stacks,biosphere
4,Acetaldehyde,1.600000e-09,kilogram,None,air::non-urban air or from high stacks,biosphere


## Checkpoint 2

Choose one of the LCIA methods already available in the installed project, ideally the climate-change indicator, and store it in `selected_method` and show its metadata.


In [ ]:
# TODO
# Example pattern:
# [m for m in bd.methods if 'climate change' in ' | '.join(m).lower()][:5]
# selected_method = ...


In [68]:
from pprint import pprint

method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

print('Method unit:', bd.Method(method).metadata.get('unit'))
pprint(bd.Method(method).metadata)

Method unit: kg CO2-Eq
{'abbreviation': 'ef-v31cg.1c397559135d78f19a1915a0ca4f626a',
 'database': 'ecoinvent-3.10-biosphere',
 'ecoinvent_version': '3.10',
 'filepath': '/Users/cmutel/Library/Application '
             'Support/EcoinventInterface/cache/ecoinvent '
             '3.10_LCIA_implementation/LCIA Implementation 3.10.xlsx',
 'geocollections': ['world'],
 'num_cfs': 183,
 'unit': 'kg CO2-Eq'}


## 7) Run your first LCA and inspect the matrices

We now combine the selected activity and method into a complete LCA calculation, and then look at the matrix objects that Brightway builds behind the scenes.

In [69]:
method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

# bw2calc.LCA is the LCA calculation engine of `bw2calc`
# it expect a functional unit demand, and an LCIA method (optional)
lca = bc.LCA({selected_activity: 1}, method)

# here, we solve the system (As=f) and obtain the inventory
lca.lci()

# and we multiply the inventory by the characterization matrix
lca.lcia()

print('Activity:', selected_activity['name'])
print('Method:', method)
print('LCA score:', lca.score)


Activity: Transport, passenger car, plugin gasoline hybrid, label-certified electricity, Large, 2018, EURO-6c
Method: ('EF v3.1', 'climate change', 'global warming potential (GWP100)')
LCA score: 0.2674738027403939


### 7.1 From the demand dict to the demand array

The argument passed to `bc.LCA(...)` is a Python dictionary: it is easy to read, but not yet in matrix form.  
After `lca.lci()`, Brightway has translated this demand into a numerical vector in product space: `lca.demand_array`.

In [70]:
print('Demand dict:')
print(lca.demand)
print()
print('Demand array shape:', lca.demand_array.shape)
print('Number of non-zero entries in demand_array:', np.count_nonzero(lca.demand_array))

Demand dict:
{343018009619595265: 1}

Demand array shape: (11747,)
Number of non-zero entries in demand_array: 1


In [71]:
demand_id = list(lca.demand.keys())[0]
demand_id

343018009619595265

In [72]:
# where is that demand in the demand vector?
demand_index = int(np.flatnonzero(lca.demand_array)[0])
demand_index

9844

In [73]:
# we have some reversed dictionaries that can 
# help us map indices (meaning position in a vector or matrix) to IDs
lca.dicts.product.reversed

{0: 343017940392607744,
 1: 343017940392607745,
 2: 343017940392607746,
 3: 343017940392607747,
 4: 343017940392607748,
 5: 343017940392607749,
 6: 343017940392607750,
 7: 343017940392607751,
 8: 343017940421967872,
 9: 343017940421967873,
 10: 343017940430356480,
 11: 343017940430356481,
 12: 343017940430356482,
 13: 343017940430356483,
 14: 343017940438745088,
 15: 343017940447133696,
 16: 343017940447133697,
 17: 343017940447133698,
 18: 343017940447133699,
 19: 343017940447133700,
 20: 343017940447133701,
 21: 343017940447133702,
 22: 343017940447133703,
 23: 343017940447133704,
 24: 343017940447133705,
 25: 343017940447133706,
 26: 343017940447133707,
 27: 343017940468105216,
 28: 343017940468105217,
 29: 343017940468105218,
 30: 343017940468105219,
 31: 343017940468105220,
 32: 343017940468105221,
 33: 343017940468105222,
 34: 343017940468105223,
 35: 343017940468105224,
 36: 343017940468105225,
 37: 343017940468105226,
 38: 343017940468105227,
 39: 343017940468105228,
 40: 34301

In [74]:
# let's check: is the position of our demand in the demand vector 
# leads to the ID we fetched from `lca.demand`?
lca.dicts.product.reversed[demand_index] == demand_id
# Cool!

True

In [75]:
# hence, we now know who the functional unit is...
demanded_product = bd.get_activity(demand_id)
pd.Series(
    {
        'demanded product index': demand_index,
        'demanded product': demanded_product['name'],
        'location': demanded_product.get('location'),
        'amount in demand_array': lca.demand_array[demand_index],
    }
)

demanded product index                                                                                                   9844
demanded product          Transport, passenger car, plugin gasoline hybrid, label-certified electricity, Large, 2018, EURO-6c
location                                                                                                                   CH
amount in demand_array                                                                                                    1.0
dtype: object

### 7.2 Dictionaries that map matrix indices back to Brightway objects

The matrices only know row and column numbers. To go back from those numbers to Brightway nodes, use the dictionaries stored on the LCA object.

Those giving `id` -> `index` live in `lca.dicts.activity`, `lca.dicts.product`, and `lca.dicts.biosphere`.

In [76]:
list(lca.dicts.product.items())[:5]

[(343017940392607744, 0),
 (343017940392607745, 1),
 (343017940392607746, 2),
 (343017940392607747, 3),
 (343017940392607748, 4)]

In [77]:
list(lca.dicts.activity.items())[:5]

[(343017940392607744, 0),
 (343017940392607745, 1),
 (343017940392607746, 2),
 (343017940392607747, 3),
 (343017940392607748, 4)]

In [78]:
list(lca.dicts.biosphere.items())[:5]

[(1, 0), (2, 1), (3, 2), (4, 3), (5, 4)]

And those giving `index` -> `id` live in `lca.dicts.activity.reversed`, `lca.dicts.product.reversed`, and `lca.dicts.biosphere.reversed`

In [79]:
selected_activity_index = lca.dicts.activity[selected_activity.id]

pd.DataFrame(
    [
        {
            'dictionary': 'activity',
            'size': len(lca.dicts.activity),
            'example matrix index': selected_activity_index,
            'example node name': bd.get_node(id=lca.dicts.activity.reversed[selected_activity_index])['name'],
        },
        {
            'dictionary': 'product',
            'size': len(lca.dicts.product),
            'example matrix index': demand_index,
            'example node name': bd.get_node(id=lca.dicts.product.reversed[demand_index])['name'],
        },
        {
            'dictionary': 'biosphere',
            'size': len(lca.dicts.biosphere),
            'example matrix index': 0,
            'example node name': bd.get_node(id=lca.dicts.biosphere.reversed[0])['name'],
        },
    ]
)

,dictionary,size,example matrix index,example node name
0,activity,11747,9844,"Transport, passenger car, plugin gasoline hybrid, label-certified electricity, Large, 2018, EURO-6c"
1,product,11747,9844,"Transport, passenger car, plugin gasoline hybrid, label-certified electricity, Large, 2018, EURO-6c"
2,biosphere,1807,0,"1,4-Butanediol"


### 7.3 The technosphere matrix and the biosphere matrix

After `lca.lci()`, Brightway has built:

- `lca.technosphere_matrix`: the technosphere matrix $A$
- `lca.biosphere_matrix`: the biosphere matrix $B$

For a demanded product, a column of the technosphere matrix shows the products consumed and produced by that unit process. A column of the biosphere matrix shows the direct elementary flows of the corresponding activity.

In [80]:
pd.DataFrame(
    [
        {'object': 'demand_array', 'shape': lca.demand_array.shape},
        {'object': 'technosphere_matrix', 'shape': lca.technosphere_matrix.shape},
        {'object': 'biosphere_matrix', 'shape': lca.biosphere_matrix.shape},
        {'object': 'supply_array', 'shape': lca.supply_array.shape},
        {'object': 'inventory', 'shape': lca.inventory.shape},
        {'object': 'characterization_matrix', 'shape': lca.characterization_matrix.shape},
        {'object': 'characterized_inventory', 'shape': lca.characterized_inventory.shape},
    ]
)

,object,shape
0,demand_array,"(11747,)"
1,technosphere_matrix,"(11747, 11747)"
2,biosphere_matrix,"(1807, 11747)"
3,supply_array,"(11747,)"
4,inventory,"(1807, 11747)"
5,characterization_matrix,"(1807, 1807)"
6,characterized_inventory,"(1807, 11747)"


In [81]:
# Let's slice the technosphere matrix to fetch all the rows (products) 
# at the column corresponding to our gasoline car transport activity.
technosphere_column = lca.technosphere_matrix[:, demand_index].tocoo()
technosphere_column

<11747x1 sparse matrix of type '<class 'numpy.float64'>'
	with 14 stored elements in COOrdinate format>

In [82]:
# let's map those indices to product names and locations
technosphere_view = pd.DataFrame(
    [
        {
            'row index': row_idx,
            'col index': demand_index,
            'product': bd.get_node(id=lca.dicts.product.reversed[row_idx])['name'],
            'location': bd.get_node(id=lca.dicts.product.reversed[row_idx]).get('location'),
            'amount in A': amount,
        }
        for row_idx, amount in zip(technosphere_column.row, technosphere_column.data)
    ]
)

technosphere_view.sort_values('amount in A', ascending=False)

# notice how inputs are stored as negative entries in the technosphere/A matrix
# except the outgoing/production exchange, which is positive and on the diagonal (where row index == col index)

,row index,col index,product,location,amount in A
12,9844,9844,"Transport, passenger car, plugin gasoline hybrid, label-certified electricity, Large, 2018, EURO-6c",CH,1.000000e+00
10,8741,9844,"Transport, freight, lorry, 16t-32t gross weight, fleet average",RER,-8.020000e-07
11,9113,9844,"Transport, freight, rail",RER,-4.810000e-06
7,7242,9844,"Passenger car, plugin gasoline hybrid, Large, 2018, EURO-6c",RER,-5.000000e-06
1,532,9844,"Carbon dioxide liquid, at plant",RER,-8.020000e-06
0,441,9844,"Brake wear emissions, passenger car",RER,-1.010000e-05
5,5087,9844,"Maintenance, passenger car",RER,-1.090000e-05
9,7818,9844,"Road wear emissions, passenger car",RER,-2.010000e-05
13,9955,9844,"Tyre wear emissions, passenger car",RER,-2.250000e-05
2,1818,9844,"Disposal, road",RER,-1.110000e-03


In [83]:
# here too, let's slice the biosphere matrix to fetch all the rows (elementary flows) 
# at the column corresponding to our gasoline car transport activity.
biosphere_column = lca.biosphere_matrix[:, selected_activity_index].tocoo()

biosphere_view = pd.DataFrame(
    [
        {
            'row index': row_idx,
            'col index': demand_index,
            'biosphere flow': bd.get_node(id=lca.dicts.biosphere.reversed[row_idx])['name'],
            'categories': '::'.join(bd.get_node(id=lca.dicts.biosphere.reversed[row_idx]).get('categories') or ()),
            'amount in B': amount,
        }
        for row_idx, amount in zip(biosphere_column.row, biosphere_column.data)
    ]
)

biosphere_view.sort_values('amount in B', ascending=False).head(12)

,row index,col index,biosphere flow,categories,amount in B
44,637,9844,"Carbon dioxide, fossil",air,1.336550e-01
43,634,9844,"Carbon dioxide, non-fossil",air,2.226952e-03
45,642,9844,"Carbon monoxide, fossil",air::non-urban air or from high stacks,1.187000e-04
9,83,9844,"Carbon monoxide, fossil",air::urban air close to ground,2.500000e-05
38,584,9844,Ammonia,air::non-urban air or from high stacks,6.340000e-06
76,1765,9844,Nitrogen oxides,air::non-urban air or from high stacks,6.290000e-06
17,223,9844,"Hydrocarbons, chlorinated",air::urban air close to ground,1.390000e-06
75,1764,9844,Nitrogen oxides,air::urban air close to ground,1.190000e-06
53,729,9844,"Hydrocarbons, chlorinated",air::non-urban air or from high stacks,1.081000e-06
4,28,9844,Ammonia,air::urban air close to ground,9.570000e-07


For this notebook, two practical readings are enough:

- in the technosphere column, the positive diagonal entry is the reference product produced by the activity
- the other entries are the technosphere inputs used by that activity
- in the biosphere column, each row is a direct emission or resource use of that activity before upstream scaling

### 7.4 Supply array and inventory

`lca.supply_array` indicates how much each activity must run to meet the final demand $$\mathbf{A}\mathbf{s} = \mathbf{f}$$  

In [84]:
# let's print the activities and the amount they must supply to satisfy the functional unit.
nonzero_supply = np.flatnonzero(lca.supply_array)

print(f"Number of supplying activities throughout the system: {nonzero_supply.shape}")

supply_view = pd.DataFrame(
    [
        {
            'activity': bd.get_node(id=lca.dicts.activity.reversed[idx])['name'],
            'location': bd.get_node(id=lca.dicts.activity.reversed[idx]).get('location'),
            'amount in supply_array': lca.supply_array[idx],
        }
        for idx in nonzero_supply
    ]
)

# let's print only the first 12 ones
supply_view.sort_values('amount in supply_array', ascending=False).head(12)

# These are 12 out of many activities along the supply chain that must provide an output to satisfy the functional unit

Number of supplying activities throughout the system: (4254,)


,activity,location,amount in supply_array
3969,"Transport, passenger car, plugin gasoline hybrid, label-certified electricity, Large, 2018, EURO-6c",CH,1.000000
3973,"Transport, transoceanic tanker",OCE,0.320744
4124,"Water, decarbonised, at plant",RER,0.301992
1516,"Gravel, crushed, at mine",CH,0.133271
891,"Electricity, certified eletricity",CH,0.125563
2215,"Natural gas, high pressure, at consumer",FR,0.122758
987,"Electricity, high voltage, certified electricity, at grid",CH,0.122143
2190,"Natural gas, high pressure, at consumer",CH,0.121089
1137,"Electricity, medium voltage, certified electricity, at grid",CH,0.120958
2358,"Natural gas, low pressure, at consumer",CH,0.119313


The inventory matrix then scales `lca.biosphere_matrix` by `lca.supply_array` $$\mathbf{G} = \mathbf{B}\,\mathrm{diag}(\mathbf{s})$$  
So summing a row in `lca.biosphere_matrix` gives a cradle-to-gate total for one pollutant.

In [85]:
noxs = [
    flow
    for flow in bd.Database('ecoinvent-3.10-biosphere')
    if flow['name'] == 'Nitrogen oxides'
]
noxs

['Nitrogen oxides' (kilogram, None, ('air', 'low population density, long-term')),
 'Nitrogen oxides' (kilogram, None, ('air', 'lower stratosphere + upper troposphere')),
 'Nitrogen oxides' (kilogram, None, ('air', 'urban air close to ground')),
 'Nitrogen oxides' (kilogram, None, ('air', 'non-urban air or from high stacks')),
 'Nitrogen oxides' (kilogram, None, ('air',))]

In [86]:
[nox.id for nox in noxs]

[4199, 4200, 4197, 4198, 4201]

But, let' be careful here: `lca.biosphere_matrix` isn't as large as the `biosphere` database, because `matrix_utils` (a sub-library) only fill in `lca.biosphere_matrix` with flows that are actuall used.

In [87]:
len(bd.Database('ecoinvent-3.10-biosphere'))

4362

In [88]:
lca.biosphere_matrix.shape

(1807, 11747)

In [89]:
nox_rows = [lca.dicts.biosphere[nox.id] for nox in noxs if nox.id in lca.dicts.biosphere]
nox_rows

[1766, 1764, 1765, 1767]

In [90]:
print("NOx flow id", " | ", "NOx index in lca.biosphere_matrix")
for flow, flow_idx in zip(noxs, nox_rows):
    print(flow.id, " | ", flow_idx)

NOx flow id  |  NOx index in lca.biosphere_matrix
4199  |  1766
4200  |  1764
4197  |  1765
4198  |  1767


Now that we have the elementary flows' `id` and position in `lca.inventory`, we can fetch the direct emissions of the activity.

In [92]:
emissions = []
for row in nox_rows:
    flow = bd.get_activity(lca.dicts.biosphere.reversed[row])
    emissions.append(
        [
            flow["name"],
            flow["categories"],
            flow["unit"],
            lca.inventory[row, selected_activity_index] # the slicing happens here
        ]
    )

print(f"Direct NOx emissions for activity with index {selected_activity_index}.")
pd.DataFrame(emissions, columns=["name", "categories", "unit", "amount"])

Direct NOx emissions for activity with index 9844.


,name,categories,unit,amount
0,Nitrogen oxides,"(air, lower stratosphere + upper troposphere)",kilogram,0.000000
1,Nitrogen oxides,"(air, urban air close to ground)",kilogram,0.000001
2,Nitrogen oxides,"(air, non-urban air or from high stacks)",kilogram,0.000006
3,Nitrogen oxides,"(air,)",kilogram,0.000000


But we can also fetch the cradle-to-gate emissions of the system...

In [93]:
emissions = []
for row in nox_rows:
    flow = bd.get_activity(lca.dicts.biosphere.reversed[row])
    emissions.append(
        [
            flow["name"],
            flow["categories"],
            flow["unit"],
            lca.inventory[row, :].sum(axis=1).item() # here, we look at all the rows
        ]
    )

print(f"Cradle-to-gate NOx emissions for the whole system.")
pd.DataFrame(emissions, columns=["name", "categories", "unit", "amount"])

Cradle-to-gate NOx emissions for the whole system.


,name,categories,unit,amount
0,Nitrogen oxides,"(air, lower stratosphere + upper troposphere)",kilogram,9.315372e-09
1,Nitrogen oxides,"(air, urban air close to ground)",kilogram,5.924025e-05
2,Nitrogen oxides,"(air, non-urban air or from high stacks)",kilogram,1.081722e-04
3,Nitrogen oxides,"(air,)",kilogram,6.790881e-05


### 7.5 Characterized inventory

After `lca.lcia()`, Brightway applies the chosen characterization factors. The matrix `lca.characterized_inventory` has the same shape as `lca.inventory`, but its values are now in impact-score units rather than physical flow units. Summing all its cells gives the LCIA score.

In [94]:
print('LCA score:', lca.score)

LCA score: 0.2674738027403939


In [95]:
print('LCA score:', np.sum(lca.characterization_matrix @ lca.inventory))

LCA score: 0.2674738027403939


Let's sum `lca.characterized_inventory` column-wise (we collapsed activities)

In [96]:
characterized_by_flow = np.asarray(lca.characterized_inventory.sum(axis=1)).ravel()

In [97]:
# let's sort them by individual score 
# and remove those that are zero 
# and keep the ten largest contributors
top_flow_indices = [
    idx for idx 
    in np.argsort(characterized_by_flow) 
    if characterized_by_flow[idx] != 0
]

In [98]:
# let's revert it, because np.argsort sorts in ascending order
top_flow_indices = top_flow_indices[::-1]

In [99]:
# and let's pick the ten largest
top_flow_indices = top_flow_indices[:10]

In [100]:
pd.DataFrame(
    [
        {
            'biosphere flow': bd.get_activity(id=lca.dicts.biosphere.reversed[idx])['name'],
            'categories': '::'.join(bd.get_activity(id=lca.dicts.biosphere.reversed[idx]).get('categories') or ()),
            'characterized contribution': characterized_by_flow[idx].round(6),
        }
        for idx in top_flow_indices
    ]
)

,biosphere flow,categories,characterized contribution
0,"Carbon dioxide, fossil",air,0.160349
1,"Carbon dioxide, fossil",air::urban air close to ground,0.059777
2,"Methane, fossil",air,0.023709
3,"Carbon dioxide, fossil",air::non-urban air or from high stacks,0.014907
4,"Methane, fossil",air::non-urban air or from high stacks,0.002418
5,Sulfur hexafluoride,air::non-urban air or from high stacks,0.002068
6,"Methane, fossil",air::urban air close to ground,0.000933
7,Dinitrogen monoxide,air::urban air close to ground,0.000713
8,Dinitrogen monoxide,air,0.000543
9,Sulfur hexafluoride,air,0.000483


We can also sum row-wise to show the contribution of activities.

In [101]:
characterized_by_activity = np.asarray(lca.characterized_inventory.sum(axis=0)).ravel()

In [102]:
# let's sort them by individual score 
# and remove those that are zero 
# and keep the ten largest contributors
top_flow_indices = [
    idx for idx 
    in np.argsort(characterized_by_activity) 
    if characterized_by_activity[idx] != 0
]

# let's revert it, because np.argsort sorts ascendingly
top_flow_indices = top_flow_indices[::-1]
# and let's pick the ten largest
top_flow_indices = top_flow_indices[:10]

pd.DataFrame(
    [
        {
            'biosphere flow': bd.get_activity(id=lca.dicts.activity.reversed[idx])['name'],
            'categories': '::'.join(bd.get_activity(id=lca.dicts.activity.reversed[idx]).get('categories') or ()),
            'characterized contribution': characterized_by_activity[idx],
        }
        for idx in top_flow_indices
    ]
)

,biosphere flow,categories,characterized contribution
0,"Transport, passenger car, plugin gasoline hybrid, label-certified electricity, Large, 2018, EURO-6c",,0.133697
1,"Natural gas, vented",,0.022444
2,"Disposal, residues, shredder fraction from manual dismantling, in MSWI",,0.007909
3,"Disposal, plastics, mixture, 15.3% water, to municipal incineration",,0.006949
4,"Sweet gas, burned in gas turbine, production",,0.004983
5,"Diesel, burned in building machine, average",,0.004566
6,"Natural gas, burned in industrial furnace 1MWth",,0.004437
7,"Pig iron, at plant",,0.003906
8,"Refinery gas, burned in furnace",,0.003890
9,"Natural gas, sweet, burned in production flare",,0.003859


## 8) Dedicated `ecoinvent` import overview

In this notebook, we only use the prepared `ecoinvent-3.10-biosphere` project archive as a shortcut for `biosphere3` and the LCIA methods.  
We do not import the `ecoinvent` technosphere database itself.

Key points:
- `ecoinvent` is not distributed in this repository.
- The full `ecoinvent` technosphere import is not run during this course.
- The overall logic is the same: create an importer, apply strategies, inspect statistics, then write the database.
- In practice, `ecoinvent` uses dedicated `bw2io` importers.


In [103]:
# Example only. Do not run in this course.
# bd.projects.set_current("ei312")
# bi.import_ecoinvent_release(
#     version="3.12", 
#     system_model="cutoff", # other options are "consequential", "apos" and "EN15804"
#     username="xxxx",
#     password="xxxx",
#     biosphere_name="biosphere" # optional, otherwise a name is chosen for you
# )

## 9) Bonus: faster solver for large systems

As the system gets larger, exact solvers may become slow, because LU factorisation needs extensive RAM that your computer may not have.  

As of `bw2calc 2.4.0`, `JacobiGMRESLCA` introduces an iterative Krylov solver, allowing for faster resolution time with reasonable approximation.

| Aspect         | `bw2calc.LCA`                                             | `JacobiGMRESLCA`                                                                          |
| -------------- | --------------------------------------------------------- | ----------------------------------------------------------------------------------------- |
| **Method**     | Directly factorizes and solves the technosphere matrix.   | Iteratively improves an approximate solution using GMRES and Jacobi preconditioning.      |
| **Strength**   | Robust, accurate and predictable for typical LCA systems. | Potentially lower memory use and faster for very large or repeated, similar calculations. |
| **Limitation** | Matrix factorization can use substantial memory.          | Convergence and accuracy depend on the matrix and solver tolerances.                      |


In [ ]:
method = [m for m in bd.methods if "GWP100" in str(m)][0]
method

In [ ]:
dataset = db.random()
dataset

In [ ]:
%%time

lca = bc.LCA({dataset:1}, method)
lca.lci()
lca.lcia()
print(lca.score)

In [ ]:
%%time

lca = bc.JacobiGMRESLCA({dataset:1}, method, rtol=1e-8)
lca.lci()
lca.lcia()
print(lca.score)

Here, the system is too small for `JacobiGMRESLCA` to make a difference, but with larger databases (e.g., Regioinvent >250K datasets) or denser ones (e.g., EXIOBASE, with >10% density), the difference can be significant.

## 10) Brief troubleshooting notes

- Project install fails: the first run of `bi.remote.install_project(...)` needs an internet connection to download the archive from `https://files.brightway.dev/`.
- Project already exists: the setup cell switches to the existing local project instead of downloading it again.
- Incomplete project contents: if the project exists but is missing `biosphere3` or the LCIA methods, delete that project or choose a fresh project name and rerun section 1.
- Workbook not found: confirm that `../../data/lci-bafu.xlsx` exists.
- `bafu-2025` missing: rerun the BAFU import cells so that the workbook is parsed and written to the project.


## Recap

After this notebook, you should now know how to:

- bootstrap a `brightway` project from a prepared archive
- import a database with `ExcelImporter`
- inspect the LCIA methods already available in a project
- import your own LCIA method
- interpret the role of strategies, matching, and statistics
- search a database for activities and inspect exchanges
- run a first LCA from demand to score
- read `demand_array`, `technosphere_matrix`, `biosphere_matrix`, `supply_array`, and `inventory`
- use the LCA dictionaries to move between matrix indices and Brightway objects
- extract cradle-to-gate pollutant totals and interpret `characterized_inventory`
- recognize how the same import logic would extend to `ecoinvent`